In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import yaml
import re
import json

In [5]:
# def _resolve_project_path() -> Path:
#     if "__file__" in globals():
#         return Path(__file__).resolve().parents[1]
#     project_path = Path.cwd().resolve()
#     if project_path.name == "src":
#         return project_path.parent
#     return project_path

# PROJECT_PATH = _resolve_project_path()
# NAN_TOKENS = {"", "nan", "n/a", "na", "null", "none"}
PROJECT_PATH = Path("../configs/London.yaml").resolve().parents[1]
# PROJECT_PATH = Path.cwd().parent
NAN_TOKENS = {"", "nan", "n/a", "na", "null", "none"}


KEEP_COLS = {
        "listings": {"id", "last_scraped", "room_type", "neighbourhood_cleansed",
            "latitude", "longitude", "price", "accommodates", "minimum_nights", "maximum_nights",
            "amenities", "bathrooms", "bathrooms_text", "bedrooms", "beds",
            "number_of_reviews", "review_scores_rating", "review_scores_value",
            "host_id", "calculated_host_listings_count","has_availability", "availability_365"},
        "calendar":{"listing_id", "date", "available", "minimum_nights", "maximum_nights"},
        "reviews": {"listing_id", "id", "date", "reviewer_id", "comments"}
}

NUMERIC_COLS = {"listings":{"minimum_nights", "maximum_nights", "review_scores_rating",
                "accommodates", "latitude", "longitude"},
                "calendar": {"minimum_nights", "maximum_nights"},
                "reviews": {}
        }

In [6]:
# Bảng tra cứu của nhóm (data/processed)
COT_CAN_LOC = {
    "listings-loc.csv": {"listing_id", "analysis_area"},
    "vung-loc.csv": {"analysis_area", "lon_min", "lat_min", "lon_max", "lat_max"},
    "reviews-loc.csv": {"review_id", "listing_id"},
}

In [7]:
def kiem_cau_truc(d: pd.DataFrame, cot_can: set, cot_so: set, moc: str = "") -> None:
        ten_moc = f" [{moc}]" if moc else ""
        thieu = cot_can - set(d.columns)
        if thieu:
                raise ValueError(f"Thiếu cột ở mốc{ten_moc}: {sorted(thieu)}")
        for cot in cot_so:
                if not pd.api.types.is_numeric_dtype(d[cot]):
                        raise TypeError(f"Cột {cot} (mốc {ten_moc}) phải có kiểu số, kiểu hiện tại là {d[cot].dtype}")

In [8]:
def check_input(config_path, root = PROJECT_PATH) -> None:
        with Path(config_path).open("r", encoding="utf-8") as f:
                config = yaml.safe_load(f)
        snapshots = config["snapshots"]
        loai_bang = ["reviews"]
        for date in snapshots:
                for bang in loai_bang:
                        path_file = root / "data" / "raw" / f"{date}" / f"{bang}.csv.gz"
                        if not path_file.exists():
                                raise FileNotFoundError(f"Không thấy tệp thô: {path_file}")
                        d = pd.read_csv(path_file, compression="gzip", nrows=0)
                        kiem_cau_truc(d, KEEP_COLS[bang], set(), f"{date}")
                        if NUMERIC_COLS[bang]:
                                kiem_cau_truc(pd.read_csv(path_file, compression="gzip", usecols=list(NUMERIC_COLS[bang])), NUMERIC_COLS[bang], NUMERIC_COLS[bang], str(date))

In [ ]:
# tiện ích : 
def doc_cfg(config_path):
    cfg = yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))
    return cfg, [str(d) for d in cfg["snapshots"]], cfg.get("qa", {}) or {}

def quy_tac(rule_name, category, condition, reason, action, affected, total,
            kpi_metric, kpi_before, kpi_after):
    """Một dòng báo cáo QA, cùng cột với reports/qa_listings_report.csv."""
    return {
        "rule_name": rule_name, "category": category, "condition": condition,
        "reason": reason, "action": action,
        "affected_rows": int(affected),
        "affected_pct": round(affected / total * 100, 2) if total else 0.0,
        "kpi_metric": kpi_metric, "kpi_before": kpi_before, "kpi_after": kpi_after,
    }

def luu_theo_moc(df, proc_dir: Path, ten_tep: str):
    """Ghi <proc_dir>/<mốc>/<ten_tep> cho từng mốc."""
    for snap, g in df.groupby("snapshot"):
        d = proc_dir / str(snap)
        d.mkdir(parents=True, exist_ok=True)
        g.to_csv(d / ten_tep, index=False, encoding="utf-8")

In [19]:
def process_reviews(config_path, root: Path = PROJECT_PATH):
        """Đọc, lọc, làm sạch, gắn cờ QA cho reviews ở cả 4 mốc và tạo báo cáo QA."""

        cfg, snapshots, qa_cfg = doc_cfg(config_path)
        min_chars = int(qa_cfg.get("review_min_chars", 5)) 
        proc_dir = root / "data" / "processed"


        with Path(config_path).open("r", encoding="utf-8") as f:
                        config = yaml.safe_load(f)
        snapshots = config["snapshots"]
        last_snap = snapshots[-1]
        proc_dir = root / "data" / "processed"
        proc_dir.mkdir(parents=True, exist_ok=True)
        review_loc = pd.read_csv(
        proc_dir / "reviews-loc.csv",
        dtype= str,
        encoding="utf-8-sig",
        )
        review_loc = review_loc.rename(columns={"listing_id": "listing_id_loc"})
        cot_loc = ["review_id", "listing_id_loc"] + [c for c in ("Mẫu LLM", "Nhãn tay") if c in review_loc.columns]


        # review_loc = pd.read_csv(proc_dir / cfg.get("reviews_ids", "reviews-loc.csv"), dtype=str,
        #                             encoding="utf-8-sig")
        # review_loc = review_loc.rename(columns={"listing_id": "listing_id_loc"})
        # cot_loc = ["review_id", "listing_id_loc"] + [c for c in ("Mẫu LLM", "Nhãn tay") if c in review_loc.columns]

        review_loc = review_loc[cot_loc].drop_duplicates("review_id")
        frames = []
        for snap in snapshots:
                path = root / "data" / "raw" / f"{snap}" / "reviews.csv.gz"
                raw = pd.read_csv(path, compression="gzip", dtype=str,
                                usecols=lambda c: c in {"listing_id", "id", "date", "reviewer_id", "comments"})
                check_input(config_path, root)
                m = raw.merge(review_loc, left_on="id", right_on="review_id", how="right")
                m["flag_not_in_snapshot"] = m["id"].isna()
                m["id"] = m["id"].fillna(m["review_id"])
                m["snapshot"] = snap
                frames.append(m)
        df = pd.concat(frames, ignore_index=True)
        n_total = len(df)
        snap_dt = pd.to_datetime(df["snapshot"])


        # # ['listing_id',
        #  'id',
        #  'date',
        #  'reviewer_id',
        #  'comments',
        #  'review_id',
        #  'listing_id_loc',
        #  'Mẫu LLM',
        #  'Nhãn tay',
        #  'flag_not_in_snapshot',
        #  'snapshot']
        


        # gán cờ cho Mẫu LLM , Nhãn tay (na = False, not na = True): 
        df['flag_LLM'] = df['Mẫu LLM'].notna()
        df['flag_human'] = df['Nhãn tay'].notna()


        # Thay "<br/>" và '\r' bằng dấu cách để 2 câu không dính vào nhau khi tìm từ khóa : 
        co_br = df['comments'].str.contains("<br/>", regex = False)
        df['comments']  = df['comments'].str.replace( r"\r?<br/>"," ", regex = True)
        n_html = df['comments'].sum()

        # Báo cáo QA:
        con_lai = df[~df["flag_not_in_snapshot"]]
        use = ~(df["flag_not_in_snapshot"])
        qa_rules = [
                quy_tac("REV1_Html_In_Comments", "Kiểu sai (Wrong Type)",
                        "comments chứa thẻ <br/> hoặc khoảng trắng thừa",
                        "Nội dung gốc lưu dạng HTML thô",
                        "Thay <br/> bằng khoảng trắng, gộp khoảng trắng -> comments",
                        n_html, n_total, "Số review còn thẻ HTML", n_total,
                        int(n_html)),

                quy_tac("REV2_Not_In_Snapshot", "Thiếu dữ liệu (Missing)",
                                "review_id trong reviews-loc.csv không có trong tệp reviews của mốc",
                                "Review tìm được ở mốc nào thì mới đưa vào mốc đó",
                                "Giữ 1 dòng, gắn cờ flag_not_in_snapshot",
                                df["flag_not_in_snapshot"].sum(), n_total, "Số review tìm thấy",
                                int(n_total), int(len(con_lai))),

                quy_tac("REV3_Usable_For_LLM", "Tổng hợp",
                                "Không rỗng, có trong mốc",
                                "Điều kiện để một review được đưa vào phân tích LLM",
                                "Cột usable_for_llm (kết hợp với Mẫu LLM)",
                                int((~use).sum()), n_total, "Số review dùng được",
                                int(n_total), int(use.sum())),
        ]
        
        report = pd.DataFrame(qa_rules)



        # Lưu kết quả:
        df["usable_for_llm"] = use

        cot_ra = [
                'snapshot',
                'listing_id',
                'review_id',
                'date',
                'reviewer_id',
                'comments',
                'listing_id_loc',
                'Mẫu LLM',
                'Nhãn tay',
                'flag_not_in_snapshot',
                'flag_LLM',
                'flag_human',
                'usable_for_llm'
                ]
        df_out = df[cot_ra]
        luu_theo_moc(df_out, proc_dir, "review_cleaned.csv")
        rep_dir = root / "reports"
        rep_dir.mkdir(parents=True, exist_ok=True)
        report.to_csv(rep_dir / "qa_reviews_report.csv", index=False, encoding="utf-8")
        return df_out, report

In [20]:

process_reviews(Path("../configs/London.yaml"), root=PROJECT_PATH)


ValueError: invalid literal for int() with base 10: 'Wonderful.  Anne is very friendly, very welcoming and very knowledgeable about the area and things to do.A lovely place to stay, comfortable, clean and friendly - Anne is a great hostess - helpful an